# FiftyOne Quickstart

Hello there! This notebook provides a brief walkthrough of [FiftyOne](https://voxel51.com/docs/fiftyone), highlighting features that will help you build better datasets and computer vision models.

We'll cover the following concepts:

- Loading a dataset [into FiftyOne](https://voxel51.com/docs/fiftyone/user_guide/dataset_creation/index.html)
- Using FiftyOne [in a notebook](https://voxel51.com/docs/fiftyone/environments/index.html#notebooks)
- Using [views](https://voxel51.com/docs/fiftyone/user_guide/using_views.html) and [the App](https://voxel51.com/docs/fiftyone/user_guide/app.html) to explore different aspects of your dataset
- [Evaluating](https://voxel51.com/docs/fiftyone/user_guide/evaluation.html) your model's predictions
- [Finding label mistakes](https://voxel51.com/docs/fiftyone/user_guide/brain.html#label-mistakes) in your datasets

## Install FiftyOne


In [ ]:
!pip install fiftyone

If you're running this in Google Colab (or any other Ubuntu 22.04 machine), you'll also need to install:

In [ ]:
!pip install fiftyone-db-ubuntu2204

## Load a dataset

Let's get started by importing the FiftyOne library:

In [ ]:
import fiftyone as fo

FiftyOne provides a number of helpful data/model resources to get you up and running on your projects. In this example, we'll load a small detection dataset from the [FiftyOne Dataset Zoo](https://voxel51.com/docs/fiftyone/user_guide/dataset_zoo/index.html).

The command below downloads the dataset from the web and loads it into a [FiftyOne Dataset](https://voxel51.com/docs/fiftyone/user_guide/basics.html) that we'll use to explore the capabilities of FiftyOne:

In [ ]:
import fiftyone.zoo as foz

dataset = foz.load_zoo_dataset("coco-2014", split='validation')

In [ ]:
print(dataset)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# !cp /root/fiftyone/coco-2014/validation/labels.json ./
# !cp -r /content/drive/MyDrive/coco2014_validation_labels.json ./
# !cp -r /content/drive/MyDrive/karpathy_test_images/ ./

Now let's launch the [FiftyOne App](https://voxel51.com/docs/fiftyone/user_guide/app.html) so we can explore the dataset visually. Right away you will see that because we are in a notebook, an embedded instance of the App with our dataset loaded has been rendered in the cell's output.

The [Session](https://voxel51.com/docs/fiftyone/api/fiftyone.core.session.html#fiftyone.core.session.Session) object created below is a bi-directional connection between your Python kernel and the FiftyOne App, as we'll see later.

In [ ]:
session = fo.launch_app(dataset)

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz
from fiftyone import ViewField as F
import json
import numpy as np
import os

bbox_area = F("bounding_box")[2] * F("bounding_box")[3]
bbox_area0 = F("bounding_box")[0]
bbox_area1 = F("bounding_box")[1]
bbox_area2 = F("bounding_box")[2]
bbox_area3 = F("bounding_box")[3]
!wget https://storage.googleapis.com/sfr-vision-language-research/datasets/coco_karpathy_test.json
ann = json.load(open('coco_karpathy_test.json','r'))
ls = ['/root/fiftyone/coco-2014/validation/data/'+ann[i]['image'].split('/')[-1] for i in range(len(ann))]

category = ['dining table', 'bed']
all_images_view = dataset.match(F("filepath").is_in(set(ls))).select_fields("ground_truth")
big_images_view = (dataset.match(F("filepath").is_in(set(ls)))
                            .select_fields("ground_truth")
                            .filter_labels("ground_truth", (bbox_area >= 0.85) & (bbox_area < 1))
                            .filter_labels("ground_truth", (bbox_area0 <= 0.01) | (bbox_area1 <= 0.01) | (bbox_area0 + bbox_area2 >= 0.99) | (bbox_area1 + bbox_area3 >= 0.99))
                            .sort_by(bbox_area, reverse=True)
                            .group_by("filepath")
                            )

print('len big_images_view: {}'.format(len(big_images_view)))

session.view = big_images_view

## select 200 images for subset
big_images_view = big_images_view.limit(200)
from collections import Counter
counter_all = Counter()
counter = Counter()
for sample in big_images_view:
    labels = set([label["label"] for label in sample.ground_truth.detections])
    counter.update(labels)

sorted_dict = dict(sorted(counter.items(), key=lambda x: x[1], reverse=True))
print(sorted_dict)

for sample in all_images_view:
  if sample.ground_truth is not None:
    labels = set([label["label"] for label in sample.ground_truth.detections])
    counter_all.update(labels)

sorted_dict_all = dict(sorted(counter_all.items(), key=lambda x: x[1], reverse=True))
print(sorted_dict_all)

ratio = {k: sorted_dict[k]/sorted_dict_all[k] for k in sorted_dict}
print(ratio)

# save json file
images = []
for sample in big_images_view:
      images.append(sample.filepath.split('/')[-1])
print(len(images))

with open(os.path.join("coco_karpathy_scale_0_85_1_colabocc_images.json"),"w") as f:
    f.write(json.dumps(images))
